# Lección 13.9 — Alucinaciones, ventana de contexto y límites actuales

> **Módulo 13 · Modelos generativos y LLMs** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 80–100 min (lección con ejercicios intercalados ≈ 55 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 11.2 (tokens), Lección 12.4 (GPT es un decoder), Lección 13.5 (cómo se entrena un LLM), Lección 13.6 (hiperparámetros de generación), Lección 13.8 (RAG)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué es una **alucinación** y por qué un LLM inventa con total seguridad.
- Distinguir entre la **confianza** de un modelo (su probabilidad) y la **verdad**.
- Entender la **ventana de contexto**: qué cabe, qué se olvida y por qué más contexto ayuda.
- Reconocer límites que vienen de la **tokenización** (contar letras, aritmética, idiomas).
- Proponer **mitigaciones**: RAG, citar fuentes, permitir decir "no sé" y verificar.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez. Carga GPT-2 small y dos funciones de ayuda: `top_tokens` (las palabras siguientes más probables) y `completar` (continúa un texto eligiendo siempre el token más probable, como en la Lección 13.6).

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()             # oculta avisos que no necesitamos

torch.manual_seed(0)                      # semilla fija: así tus números coinciden con las respuestas
tok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2").eval()

def top_tokens(prompt, k=5):
    ids = tok(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        probs = torch.softmax(gpt2(ids).logits[0, -1], dim=-1)   # probabilidad de cada token siguiente
    p, i = probs.topk(k)
    return [(tok.decode(j), round(x.item(), 3)) for x, j in zip(p, i)]

def completar(prompt, n_tokens=25):
    ids = tok(prompt, return_tensors="pt")
    out = gpt2.generate(**ids, max_new_tokens=n_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.input_ids.shape[1]:])

plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el estudiante que nunca dice "no sé"

Imagina un examen oral con un estudiante que **leyó muchísimo**, pero al que **jamás** premiaron por decir *"no lo sé"*. Le preguntas por un autor que no existe:

> 👨‍🏫 *"¿Qué opinas de la obra de Teodoro Plumvalle?"*
> 🧑‍🎓 *"Plumvalle fue clave en el realismo mágico de los años 60; su novela más conocida…"*

No miente a propósito: **produce lo que suena a una buena respuesta**. Eso es una **alucinación** (*hallucination*): un texto fluido y seguro que no se apoya en hechos.

Y una segunda limitación: el estudiante responde con una **pizarra de tamaño fijo** delante. Todo lo que le digas tiene que caber en ella; cuando se llena, lo más antiguo se borra. Esa pizarra es la **ventana de contexto** (*context window*).

### ✋ Ejercicio 1 — Calentamiento
Recuerda la Lección 13.5: ¿con qué objetivo se pre-entrena un LLM? Con esa pista, explica en una frase por qué un LLM **prefiere inventar** a quedarse callado.

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Se entrena para **predecir el siguiente token de forma plausible**, no para decir la verdad. En los textos de internet casi nunca aparece *"no lo sé"* después de una pregunta: aparece una respuesta. Así que lo más "probable" es producir algo con forma de respuesta, exista o no la información. El ajuste por instrucciones y el RLHF reducen el problema, pero no lo eliminan.
</details>

---
## 2. 📖 El concepto: alucinar con seguridad

Le preguntaremos a GPT-2 por **Elena Varkoni**, una química peruana que **no existe** (la inventamos para esta lección).

### 🔮 Ejercicio 2 — ¿Qué dirá de alguien que no existe?
Completamos *"Dr. Elena Varkoni, the famous Peruvian chemist, is best known for"*. ¿Crees que GPT-2 dirá algo como "no tengo información"? ¿O inventará?

✍️ *Mi predicción:*

In [ ]:
print(repr(completar("Dr. Elena Varkoni, the famous Peruvian chemist, is best known for")))

<details><summary><b>👀 Ver respuesta</b></summary>

Inventa sin dudar: *"her work on the chemical composition of the earth's crust…"*. Es una respuesta **plausible** (una química que estudia la corteza terrestre suena bien) y completamente **falsa**: la persona no existe. El modelo no tiene ningún mecanismo para notar que nunca oyó hablar de ella.
</details>

### 📖 Confianza ≠ verdad
¿Al menos el modelo "duda" cuando inventa? Comparemos la probabilidad que da a la palabra siguiente en una frase **verdadera** y en una **inventada**:

### 🔮 Ejercicio 3 — ¿Dónde está más seguro?
1. *"The Eiffel Tower is located in the city of"* → la respuesta correcta es *Paris*.
2. *"Dr. Elena Varkoni, the famous Peruvian chemist, won the Nobel Prize in"* → no hay respuesta correcta: nunca ganó nada.

¿En cuál crees que GPT-2 da **más probabilidad** a su palabra favorita?

✍️ *Mi predicción:*

In [ ]:
for prompt in ["The Eiffel Tower is located in the city of",
               "Dr. Elena Varkoni, the famous Peruvian chemist, won the Nobel Prize in",
               "The Zorblax Institute of Marine Botany was founded in the year"]:   # ✏️ agrega los tuyos
    print(prompt)
    print("   →", top_tokens(prompt), "\n")

<details><summary><b>👀 Ver respuesta</b></summary>

¡Al revés de lo que uno esperaría!
- Para la Torre Eiffel, " Paris" es la favorita, pero solo con ≈**6 %**; London, Amsterdam y Berlin le siguen de cerca.
- Para la química inventada, " Chemistry" tiene ≈**50 %**: está **mucho más segura** de un dato que no existe.
- Para el instituto inventado, el año " 2000" aparece con ≈13 %.

¿Por qué? "Química → Nobel de Química" es un patrón lingüístico muy frecuente; "Eiffel → Paris" es un **hecho** que GPT-2 small (un modelo pequeño) apenas recuerda. **La probabilidad mide qué tan típico es el texto, no qué tan cierto es.** Los modelos grandes están algo mejor "calibrados", pero el problema de fondo persiste.
</details>

### 📖 Por qué alucinan los LLMs

| Causa | Explicación |
|---|---|
| **Objetivo de entrenamiento** | Se premia el texto plausible, no el verdadero (Lección 13.5) |
| **Conocimiento comprimido** | Miles de millones de hechos en un número fijo de pesos: los raros se recuerdan mal o se mezclan |
| **Nunca aprendió a callarse** | En los datos, después de una pregunta casi siempre viene una respuesta |
| **Muestreo** | Con temperatura alta (Lección 13.6) a veces se elige un token poco probable… y el resto del texto lo justifica |
| **Fecha de corte** | No sabe nada posterior a sus datos, pero puede "inventar" el presente |
| **Bola de nieve** | Una vez que escribió un dato falso, lo sigue usando para ser coherente |

---
## 3. 📖 La ventana de contexto: la pizarra de tamaño fijo

Un Transformer solo puede mirar un número máximo de tokens a la vez (Lección 12.3: las posiciones también se codifican). En GPT-2 son **1,024 tokens**; en los LLMs actuales, de decenas de miles a más de un millón. Todo cuenta: tus instrucciones, los documentos que pegas, la conversación previa **y** la respuesta que va escribiendo.

### 🔮 Ejercicio 4 — ¿Qué pasa si te pasas?
Le daremos a GPT-2 una entrada de **1,100** tokens. ¿Qué crees que ocurrirá: la lee entera, ignora el final, o falla?

✍️ *Mi predicción:*

In [ ]:
print("Ventana de contexto de GPT-2:", gpt2.config.n_positions, "tokens")

entrada_larga = torch.randint(0, 1000, (1, 1100))         # 1,100 tokens cualesquiera
try:
    gpt2(entrada_larga)
    print("La procesó sin problemas")
except Exception as e:
    print("❌ Error:", type(e).__name__, "-", e)

<details><summary><b>👀 Ver respuesta</b></summary>

**Falla** con un `IndexError`: el modelo solo tiene "posiciones" aprendidas para 1,024 tokens y no sabe qué hacer con la posición 1,025. Por eso las aplicaciones **recortan** la entrada (normalmente se quedan con lo más reciente) o la resumen. El modelo nunca se queja: simplemente **no ve** lo que se recortó.
</details>

### 🔮 Ejercicio 5 — El nombre olvidado
Al inicio decimos *"My name is Sofia."*, luego agregamos un relleno (una frase aburrida repetida N veces) y al final pedimos *"…my name is"*. Recortamos a los **últimos 1,024 tokens**, como haría una aplicación.

Con N = 5, 50 y 120 repeticiones, ¿cuándo dejará de recordar "Sofia"?

✍️ *Mi predicción:*

In [ ]:
relleno = " The weather was mild and the streets were quiet that day."
id_sofia = tok(" Sofia").input_ids[0]                     # primer token de " Sofia"

for repeticiones in [5, 50, 120]:                          # ✏️ prueba 80 y 100
    texto = "My name is Sofia." + relleno * repeticiones + " As I said before, my name is"
    ids = tok(texto, return_tensors="pt").input_ids
    n_tokens = ids.shape[1]
    ids = ids[:, -1024:]                                   # la aplicación recorta: solo cabe lo último
    with torch.no_grad():
        probs = torch.softmax(gpt2(ids).logits[0, -1], dim=-1)
    print(f"{repeticiones:>3} repeticiones · {n_tokens:>5} tokens → P(Sofia) = {probs[id_sofia]:.4f} · "
          f"favorita: {tok.decode(probs.argmax())!r}")

<details><summary><b>👀 Ver respuesta</b></summary>

- Con 5 y 50 repeticiones (74 y 614 tokens), la probabilidad de "Sofia" es ≈**0.76** y ≈**0.74**: lo recuerda muy bien, aunque esté a 600 tokens de distancia. Dentro de la ventana, los Transformers **copian** con mucha facilidad (la atención, Lección 12.2, "mira" directamente hacia atrás).
- Con 120 repeticiones (1,454 tokens) la frase con el nombre **quedó fuera** del recorte: P(Sofia) cae a ≈0.0001 y el modelo propone *" John"*. No es que lo recuerde mal: **nunca lo vio**.

Esto pasa en los chats largos: si la conversación supera la ventana, lo primero que dijiste desaparece en silencio.
</details>

### 📖 Más contexto = mejores predicciones (si cabe)
Hagamos un experimento limpio: una secuencia de 200 tokens **al azar**, repetida dos veces. La primera copia es imposible de predecir; la segunda, en teoría, es trivial si el modelo **mira hacia atrás**.

### 🔮 Ejercicio 6 — La segunda copia
1. ¿Qué pérdida media esperas en la **primera** copia? (Pista: la pérdida inicial de la Lección 13.5 con un vocabulario de 50,257 tokens.)
2. ¿Y en la **segunda**?
3. Si solo dejamos que el modelo vea los últimos **50**, **150** o **250** tokens, ¿en cuál podrá copiar?

✍️ *Mis predicciones:*

In [ ]:
torch.manual_seed(0)
patron = torch.randint(1000, 20000, (200,))                # 200 tokens al azar
secuencia = torch.cat([patron, patron])                    # ...repetidos dos veces

with torch.no_grad():
    logp = torch.log_softmax(gpt2(secuencia[None]).logits[0], dim=-1)
perdidas = -logp[:-1].gather(1, secuencia[1:, None])[:, 0]  # pérdida de cada token
print(f"Pérdida media en la 1.ª copia: {perdidas[:199].mean():.2f}")
print(f"Pérdida media en la 2.ª copia: {perdidas[199:].mean():.2f}")

plt.plot(perdidas.numpy(), lw=0.8)
plt.axvline(199, color="red", ls="--", label="empieza la 2.ª copia")
plt.xlabel("posición"); plt.ylabel("pérdida del token"); plt.legend(); plt.title("Copiar del contexto"); plt.show()

for ventana in [50, 150, 250]:                              # ✏️
    ps = []
    for pos in range(260, 300, 4):                          # tokens de la 2.ª copia
        ctx = secuencia[max(0, pos - ventana):pos]           # el modelo solo ve los últimos `ventana` tokens
        with torch.no_grad():
            ps.append(-torch.log_softmax(gpt2(ctx[None]).logits[0, -1], -1)[secuencia[pos]].item())
    print(f"ventana de {ventana:>3} tokens → pérdida media en la 2.ª copia: {np.mean(ps):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. Primera copia: ≈**11.6**, del orden de ln(50,257) ≈ 10.8 (incluso un poco peor: tokens al azar son *más* raros que "uniformes" para el modelo).
2. Segunda copia: ≈**0.13**: predice casi a la perfección, porque **copia** de lo que ya vio. En la gráfica, la pérdida se desploma justo en la línea roja.
3. Con ventanas de 50 o 150 tokens la pérdida sigue en ≈**11**: la primera copia del token que toca está a 200 posiciones y **no cabe** en la ventana. Con 250 tokens cae a ≈**0.01**.

Moraleja: el contexto es **la memoria de trabajo** del modelo. Lo que está dentro se usa muy bien; lo que queda fuera, no existe. Por eso RAG (Lección 13.8) pone la información relevante **dentro** de la ventana.
</details>

---
## 4. 📖 Límites que vienen de los tokens

El modelo no ve letras ni números: ve **tokens** (Lección 11.2). Eso explica varios fallos que parecen absurdos.

In [ ]:
for palabra in ["strawberry", " strawberry", "hippopotamus", " murciélago", " extraordinariamente"]:   # ✏️
    print(f"{palabra!r:<24} → {[tok.decode(t) for t in tok(palabra).input_ids]}")

en = "The children played in the park until the sun went down and their parents called them home for dinner."
es = "Los niños jugaron en el parque hasta que se puso el sol y sus padres los llamaron a casa para cenar."
print(f"\nInglés : {len(en.split())} palabras → {len(tok(en).input_ids)} tokens")
print(f"Español: {len(es.split())} palabras → {len(tok(es).input_ids)} tokens")

### 🔮 Ejercicio 7 — Contar y sumar
1. Si para el modelo *" strawberry"* es **un solo token**, ¿qué tan fácil le resulta contar cuántas "r" tiene?
2. Le damos ejemplos de sumas y pedimos otras. ¿Acertará GPT-2 en `2 + 3`? ¿Y en `4821 + 3907`?

✍️ *Mis predicciones:*

In [ ]:
ejemplos = "1 + 1 = 2\n3 + 4 = 7\n10 + 5 = 15\n"
for a, b in [(2, 3), (12, 25), (48, 37), (4821, 3907)]:   # ✏️
    respuesta = completar(f"{ejemplos}{a} + {b} =", n_tokens=4).split("\n")[0]
    print(f"{a} + {b} = {a + b:<5} │ GPT-2 dice:{respuesta!r}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. Muy difícil: *" strawberry"* llega como **una sola pieza** (sin espacio se parte en `st` + `raw` + `berry`). El modelo nunca "ve" las letras sueltas; solo podría saber cuántas "r" hay si lo memorizó de algún texto. Por eso hasta modelos enormes fallaban en esto.
2. **Falla en todas**: dice 4 para `2 + 3`, 30 para `12 + 25`, 50 para `48 + 37`, y para `4821 + 3907` se pone a repetir los ejemplos. GPT-2 imita el **formato** ("número = número"), pero no calcula. Los números grandes, además, se trocean de formas irregulares (`4821` puede ser `48` + `21`).

Y fíjate en el idioma: la misma frase usa **20 tokens en inglés y 34 en español**. En español pagas más (las APIs cobran por token), cabe menos texto en la ventana y el modelo suele rendir peor.

Mitigación típica: dejar que el modelo use **herramientas** (una calculadora, código) en lugar de "calcular de memoria": es la idea de los agentes de la Lección 13.8.
</details>

---
## 5. 📖 Mitigaciones: cómo reducir las alucinaciones

| Estrategia | Idea | Limitación |
|---|---|---|
| **RAG** (Lección 13.8) | Poner en el contexto documentos relevantes y pedir que responda **con ellos** | Si la recuperación falla, sigue inventando |
| **Citar fuentes** | Exigir que cada afirmación diga de qué documento sale | Puede inventar citas: hay que comprobarlas |
| **Permitir "no sé"** | Instrucciones y datos de ajuste que premian reconocer la ignorancia | Puede volverse demasiado cauto |
| **Temperatura baja** (Lección 13.6) | Menos tokens improbables | No arregla lo que el modelo "cree" saber |
| **Verificación** | Otro paso (otro modelo, una búsqueda, una persona) revisa la respuesta | Más costo y tiempo |
| **Consistencia** | Preguntar varias veces: si las respuestas cambian, desconfía | En modelos pequeños, todo varía |
| **Herramientas** | Calculadora, buscador, base de datos | Hay que diseñar bien qué puede hacer el modelo (seguridad, Lección 15.5) |

Probemos la primera: darle a GPT-2 el dato correcto **dentro del contexto**.

### 🔮 Ejercicio 8 — Contexto contra memoria
Antes de la pregunta pegamos una "ficha" (un documento recuperado): *Elena Varkoni es una chef peruana; nunca ganó un Nobel, pero ganó el premio Golden Spoon en 2019.*

¿Seguirá diciendo "Nobel"? ¿Qué palabra crees que preferirá ahora?

✍️ *Mi predicción:*

In [ ]:
ficha = ("Elena Varkoni is a Peruvian chef. She has never won a Nobel Prize, "
         "but in 2019 she won the Golden Spoon award for cooking.\n")
pregunta = "Question: What award did Elena Varkoni win?\nAnswer: She won the"

print("Sin ficha:", top_tokens(pregunta))
print("Con ficha:", top_tokens(ficha + pregunta))

<details><summary><b>👀 Ver respuesta</b></summary>

- **Sin ficha**, la favorita es un genérico " award" (≈12 %) y " Nobel" anda en ≈2 %.
- **Con ficha**, la favorita pasa a ser **" Golden"** (≈22 %): el modelo **usa** el documento que le dimos. Es exactamente la idea de RAG (Lección 13.8).
- Pero ojo: " Nobel" **sube** a ≈10 %, porque la palabra aparece en la ficha ("never won a Nobel Prize") y el modelo, pequeño, copia palabras sin entender bien la negación.

El contexto ayuda muchísimo, pero no es magia: un modelo débil puede leer mal los documentos. Los modelos grandes son mucho mejores leyendo, y aun así conviene pedirles que **citen** el fragmento que usaron.
</details>

---
## 6. 📖 Otros límites actuales (para tener criterio)

- **Fecha de corte:** el modelo no sabe lo que pasó después de sus datos.
- **Razonamiento frágil:** acierta un problema y falla una variante con otros números.
- **Sesgos:** reproduce estereotipos de los textos con que se entrenó (Lección 15.4).
- **Inyección de instrucciones** (*prompt injection*): un documento recuperado puede contener órdenes ("ignora lo anterior y…") que el modelo obedece (Lección 15.5).
- **"Perdido en el medio"** (*lost in the middle*): con contextos muy largos, la información del medio se aprovecha peor que la del principio o el final.
- **Costo y latencia:** cada token cuesta cómputo; contextos largos son caros.

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué hacer |
|---|---|---|
| Cita artículos, leyes o enlaces que no existen | Alucinación: el formato "cita" es muy típico | Verificar cada fuente; usar RAG con documentos reales |
| "Lo dijo con mucha seguridad, debe ser cierto" | Confianza ≠ verdad (Ejercicio 3) | Pedir fuentes y comprobarlas |
| En chats largos olvida lo que dijiste al principio | Se superó la ventana de contexto y se recortó | Resumir la conversación, repetir lo importante, RAG |
| Errores en cuentas y en contar letras | Tokenización; no calcula, imita | Darle una calculadora o que escriba código |
| Respuestas peores en español que en inglés | Menos datos y más tokens por palabra | Probar el idioma real de tus usuarios antes de lanzar |
| Pegas un documento enorme y la respuesta ignora una parte | Recorte silencioso o "perdido en el medio" | Contar tokens; poner lo importante al principio o al final |

### 🔮 Ejercicio 9 — Diagnóstico
Un abogado usa un chatbot para preparar un escrito. El bot cita tres sentencias con número de expediente, fecha y juez. ¿Qué probabilidad hay de que sean reales? ¿Qué debería hacer el abogado?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**No se puede saber sin comprobarlo**, y es muy posible que alguna sea inventada: una cita legal tiene un formato muy predecible (número, fecha, juez) que el modelo sabe imitar a la perfección. Ya ocurrió en la vida real: abogados sancionados por presentar sentencias falsas generadas por un chatbot. Lo correcto: buscar cada sentencia en una base de datos oficial antes de usarla, o usar una herramienta con RAG sobre jurisprudencia real que muestre el documento fuente.
</details>

---
## 8. 🧠 Resumen en 5 puntos
1. Una **alucinación** es un texto fluido y seguro que no se apoya en hechos; nace de un entrenamiento que premia lo **plausible**, no lo verdadero.
2. **Confianza ≠ verdad:** GPT-2 estaba más seguro del "Nobel" de una química inventada que de que la Torre Eiffel está en París.
3. La **ventana de contexto** es la memoria de trabajo: dentro, el modelo copia y usa la información muy bien; fuera, no existe (y el recorte es silencioso).
4. Muchos fallos "tontos" (contar letras, sumar, rendir peor en español) vienen de los **tokens**.
5. Se mitiga con **RAG**, fuentes verificables, permiso para decir **"no sé"**, herramientas y verificación; ninguna medida elimina el problema por completo.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** ¿Cuál es la causa de fondo de las alucinaciones?
- a) Un error de programación · b) El modelo se entrena para producir texto plausible, no verdadero · c) La GPU se calienta · d) La temperatura siempre es 0

**2.** Un LLM da una respuesta con probabilidad 0.95. Eso significa que:
- a) Es verdadera con 95 % de seguridad · b) Es un texto muy típico para el modelo, sea cierto o no · c) El modelo la buscó en internet · d) Está citando una fuente

**3.** GPT-2 tiene una ventana de 1,024 tokens. Pegas un documento de 3,000 tokens y la aplicación conserva los últimos 1,024. ¿Qué pasa con el principio del documento?
- a) El modelo lo resume · b) El modelo lo recuerda vagamente · c) El modelo no lo ve en absoluto · d) Se guarda en los pesos

**4.** ¿Por qué un LLM suele fallar al contar las letras de una palabra?
- a) Porque no sabe contar hasta 10 · b) Porque ve tokens (trozos de palabras), no letras · c) Porque la temperatura es alta · d) Porque la palabra está en inglés

**5.** ¿Qué mitigación pone la información correcta **dentro de la ventana de contexto**?
- a) Bajar la temperatura · b) Cuantizar el modelo · c) RAG · d) Aumentar `max_new_tokens`

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** El objetivo de pre-entrenamiento (siguiente token) premia la plausibilidad.
2. **b)** La probabilidad mide qué tan esperable es el texto para el modelo, no si es cierto (Ejercicio 3).
3. **c)** Lo recortado no existe para el modelo (Ejercicio 5).
4. **b)** *" strawberry"* es un solo token para GPT-2.
5. **c)** RAG recupera documentos y los pega en el contexto (Lección 13.8).
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Si los LLMs alucinan, ¿por qué se usan tanto? Piensa en tareas donde una alucinación es **barata** (y fácil de detectar) y otras donde es **carísima**.

**B. Caso de análisis 💼** Un hospital quiere un asistente que responda preguntas de los pacientes sobre sus medicamentos. Diseña, en 4–5 viñetas, cómo reducirías el riesgo de alucinaciones. Incluye al menos una medida técnica y una **no técnica**.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Porque en muchas tareas la alucinación es barata o fácil de detectar: redactar un borrador que una persona revisa, generar ideas, resumir un texto que tienes delante, escribir código que luego se prueba. Es carísima cuando nadie verifica y el error tiene consecuencias: medicina, derecho, finanzas. La pregunta útil no es "¿alucina?" (sí), sino **"¿qué pasa si alucina y quién lo detectaría?"**.

**B.** Un diseño razonable:
- **RAG** solo sobre los prospectos oficiales y la ficha del paciente; prohibir responder fuera de esas fuentes.
- **Citar** el párrafo exacto del documento en cada respuesta, para que pueda revisarse.
- Instrucción y pruebas para que diga **"no lo sé, consulte a su médico"** cuando la fuente no responde la pregunta.
- **Temperatura baja** y respuestas cortas.
- (No técnica) Un aviso visible de que no reemplaza al médico, revisión periódica de conversaciones por farmacéuticos y un canal para reportar errores. Medir la tasa de alucinación con un conjunto de preguntas de prueba antes de lanzar (Lección 15.1).
</details>

### 9.3 🧩 Práctica integradora: tu detector casero de alucinaciones

Vas a comprobar si alguna "señal" del propio modelo sirve para detectar cuándo inventa. Cada paso tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Preguntas reales e inventadas.** Ejecuta la celda: para cada pregunta mide la probabilidad de la palabra favorita de GPT-2.

In [ ]:
preguntas = {
    # (texto, ¿tiene respuesta real?)
    "The Eiffel Tower is located in the city of": True,
    "The largest ocean on Earth is the": True,
    "Barack Obama was born in the state of": True,
    "Dr. Elena Varkoni, the famous Peruvian chemist, won the Nobel Prize in": False,
    "The Zorblax Institute of Marine Botany was founded in the year": False,
    "The 2031 World Cup was won by": False,
}
for p, real in preguntas.items():
    favorita, prob = top_tokens(p, k=1)[0]
    print(f"{'REAL ' if real else 'FALSA'} · P={prob:.3f} · favorita {favorita!r:<12} │ {p}")

<details><summary><b>👀 Ver respuesta</b></summary>

La señal **no sirve**, e incluso funciona al revés:
- Las preguntas reales tienen probabilidades bajas: Paris ≈0.06, " largest" ≈0.05 (ni siquiera es una respuesta) y Obama → " Maryland" ≈0.16, que además es **falso** (nació en Hawái).
- Las inventadas tienen probabilidades iguales o más altas: " Chemistry" ≈**0.50**, " 2000" ≈0.13.

Si pusieras un umbral del tipo "si P < 0.1, decir *no sé*", el modelo diría "no sé" en dos de las preguntas reales y respondería con total seguridad sobre la química inventada. En GPT-2 small, la probabilidad casi no dice nada sobre la verdad. En modelos grandes la señal es algo mejor, pero nunca es una garantía.
</details>

**Paso 2 🔮 — Consistencia.** Otra idea: pregunta 6 veces con azar (`do_sample=True`). Si el modelo "sabe", debería repetir la misma respuesta. ¿Será más consistente con las preguntas reales?

In [ ]:
def muestras(prompt, n=6, n_tokens=4):
    torch.manual_seed(0)
    ids = tok(prompt, return_tensors="pt")
    out = gpt2.generate(**ids, max_new_tokens=n_tokens, do_sample=True, num_return_sequences=n,
                        pad_token_id=tok.eos_token_id)
    return [tok.decode(o[ids.input_ids.shape[1]:]).strip() for o in out]

for p in ["The Eiffel Tower is located in the city of", "Barack Obama was born in the state of",
          "Dr. Elena Varkoni was born in the city of"]:     # ✏️
    print(p, "\n   →", muestras(p))

<details><summary><b>👀 Ver respuesta</b></summary>

**No, en GPT-2 small todo es inconsistente**: para la Torre Eiffel salen Ghent, Barcelona, London, Berlin, New Orleans…; para Obama, Michigan, Wyoming, Minnesota, Illinois (dos veces), Oklahoma; para la química inventada, ciudades distintas cada vez. Este modelo es tan pequeño que **no recuerda con firmeza ni los hechos reales**, así que la consistencia no distingue lo real de lo inventado.

En modelos grandes la técnica sí funciona mejor (es la idea de *SelfCheckGPT*): los hechos bien aprendidos se repiten, los inventados cambian. Moraleja de método: **una técnica de detección hay que medirla con tu modelo y tus preguntas** antes de confiar en ella.
</details>

**Paso 3 ✏️ — La cura: contexto.** Escribe tu propia "ficha" con un dato inventado (por ejemplo, la ciudad donde nació Elena Varkoni) y comprueba si GPT-2 la usa. Cambia `FICHA` y `PREGUNTA`.

In [ ]:
FICHA = "Elena Varkoni was born in Arequipa, a city in the south of Peru.\n"     # ✏️
PREGUNTA = "Elena Varkoni was born in the city of"                                  # ✏️
print("Sin ficha:", top_tokens(PREGUNTA))
print("Con ficha:", top_tokens(FICHA + PREGUNTA))

<details><summary><b>👀 Ver respuesta</b></summary>

Sin ficha, el modelo reparte la probabilidad entre letras sueltas (" B", " P", " L"… ≈2 % cada una): no tiene ni idea. Con la ficha, la favorita pasa a ser **" Are"** (≈35 %), el primer token de "Arequipa": **copia el dato del contexto**. De nuevo, lo que está dentro de la ventana de contexto pesa mucho más que la "memoria" de los pesos. (Si escribiste otra ficha, deberías ver algo parecido: el primer trozo de tu dato sube al primer lugar o cerca.)
</details>

**Paso 4 ✋ — Tu informe.** En 4 frases, para un gerente que quiere "un chatbot que nunca invente": ¿qué le prometerías y qué no?

✍️ *Mi informe:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo: *"Ningún modelo de lenguaje actual garantiza no inventar: está hecho para escribir texto que suene bien, y en nuestras pruebas estaba más seguro de un dato falso que de uno verdadero. Lo que sí podemos prometer es reducir mucho el riesgo: el bot responderá a partir de nuestros documentos, mostrará de dónde sacó cada dato y dirá 'no lo sé' cuando no encuentre la respuesta. Mediremos con preguntas de prueba cuántas veces se equivoca antes de lanzarlo y revisaremos conversaciones reales cada mes. En temas delicados, la respuesta final la dará siempre una persona."*
</details>

---
## 📚 Para profundizar
- 📄 **Artículo:** Ji et al. (2023), *Survey of Hallucination in Natural Language Generation*.
- 📄 **Artículo:** Liu et al. (2023), *Lost in the Middle: How Language Models Use Long Contexts*.
- 📄 **Artículo:** Manakul et al. (2023), *SelfCheckGPT*: detectar alucinaciones por consistencia (la idea del Paso 2).
- 🎮 **Interactivo:** el "Tokenizer" de OpenAI o de Hugging Face: pega textos en español e inglés y cuenta los tokens.

---
⬅️ **Anterior:** [13.8 Prompting, RAG y agentes: sacarle partido a un LLM sin reentrenarlo](13.8-prompting-rag-y-agentes.ipynb) · ➡️ **Siguiente:** [13.10 Proyecto del módulo: El asistente de Cóndor Air ✈️](13.10-proyecto-del-modulo.md)